# 00aii · Cell type annotation (**Monocytes, macrophages, cDC2s** analytical manifold, acrosslifespan/no nEMC)

**Sections:**
1. Load integrated .h5ad object from `01ai_scvi_integration_monomacscDC2_acrosslifespan_noEMC.ipynb`
2. View existing metadata via UMAP
3. Plot previous cell type annotations
4. Optional: re-cluster specific Leiden clusters
5. TF-IDF: identify quick markers across Leiden clusters
6. Visualise known markers via dotplots (UMAP visualisation of the same markers, and others, was paralleled but omitted here for space)
7. Manually map cell types to Leiden clusters along with accompanying cell ontology
8. Save .h5ad
9. Contextualise cell type annotations within metadata with bar plots

*Note:* Several rounds of iteration within Sections 4-6 was performed prior to finalising cell type annotations.

## Configuration

In [ ]:
import os, sys

# ── Paths ─────────────────────────────────────────────────────────────────────
MAIN_DIR        = "/nfs/team292/projects/PanTissue/"
INPUT_H5AD      = os.path.join(MAIN_DIR, "results/temp/01_integration/integrated_scvi_acrosslifespan_noHormones_immune_monomacscDC2.h5ad")
OUTPUT_DIR      = os.path.join(MAIN_DIR, "results/temp/02_annotation/immune/acrosslifespan/noHormones/00_monomacscDC2/")
MARKERS_PY      = os.path.join(MAIN_DIR, "code/working/ck18/markers.py")
MARKERS_IMMUNE_PY = os.path.join(MAIN_DIR, "code/working/ck18/markers_immune.py")
FORANNOTION_H5AD   = os.path.join(OUTPUT_DIR, "forannotation_acrosslifespan_noHormones_immune_monomacscDC2.h5ad")
ANNOTATED_H5AD   = os.path.join(OUTPUT_DIR, "annotated_acrosslifespan_noHormones_immune_monomacscDC2.h5ad")
# ONTOLOGY_CSV  = os.path.join(MAIN_DIR, "data/freeze/ontology/HCA_annotation_ontology_L1_L4_postnatal_immune.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Column names in obs ───────────────────────────────────────────────────────
DATASET_COL  = "dataset"
DONOR_COL    = "Donor_id"
SAMPLE_COL   = "sample"
ORGAN_COL    = "Organ"
LEIDEN_COL   = "leiden"

# QC columns to use in boxplots — adjust to what is present in your obs
QC_COLS = [
    "n_genes_by_counts",
    "n_genes",
    "percent_mito",
    # "phase",
    "health_score",
    'stress_score_vandenBrink',
    "doublet_scores",
]

# ── Misc ──────────────────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── TF-IDF parameters ─────────────────────────────────────────────────────────
TFIDF_DOWNSAMPLE_N = 100    # cells per celltype_leiden group
TFIDF_N_MARKERS    = 10
TFIDF_FDR          = 0.01
TFIDF_EXPRESS_CUT  = 0.9

## Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import importlib.util
import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import anndata as ad
import scanpy as sc

sys.path.append('/nfs/team292/projects/PanTissue/code/working/lg18/utils/')
from panatlas_utils import quick_markers, Barplot, filterTFIDF, plotUMAP_per_value, get_available_markers, plot_obs_facets

sc.settings.figdir = os.path.join(OUTPUT_DIR, "figures/")
sc.settings.verbosity = 3
sc.set_figure_params(dpi=100, frameon=False, figsize=(6, 5))

print(f"anndata : {ad.__version__}")
print(f"scanpy  : {sc.__version__}")

# 1. Load integrated object

In [ ]:
# adata = ad.read_h5ad(INPUT_H5AD)
adata = ad.read_h5ad(FORANNOTION_H5AD)
print(adata)
print("\nobs columns:", adata.obs.columns.tolist())

In [ ]:
adata.obs.head(1)

# 2. Overview UMAPs

In [ ]:
# Fix typos
adata.obs["Menstrual_stage"] = (
    adata.obs["Menstrual_stage"]
    .replace("Unkown", "Unknown")
    .astype("category") # Ensure it stays/becomes a category for Scanpy
)
# Fix typos
adata.obs["Menstrual_stage"] = (
    adata.obs["Menstrual_stage"]
    .replace("Postmenopausal", "Postmenopause")
    .astype("category") # Ensure it stays/becomes a category for Scanpy
)
# Fix typos
adata.obs["Menstrual_stage"] = (
    adata.obs["Menstrual_stage"]
    .replace("Secreotry", "Secretory")
    .astype("category") # Ensure it stays/becomes a category for Scanpy
)
# Fix typos
adata.obs["Organ_part"] = (
    adata.obs["Organ_part"]
    .replace("Mentrual fluid", "Menstrual fluid")
    .astype("category") # Ensure it stays/becomes a category for Scanpy
)
# Fix typos
adata.obs["Menstrual_stage_short"] = (
    adata.obs['Menstrual_stage'].astype(str).str.split(" ").str[0]
)
adata.obs["Menstrual_stage_short"] = (
    adata.obs["Menstrual_stage_short"]
    .replace("Not", "Not applicable")
)

print(adata.obs["Menstrual_stage_short"].value_counts())

# Rename menstrual fluid
mask = (adata.obs['dataset'] == "uterus_adult_menstrualfluid_sanger-denoised") & (adata.obs['Menstrual_stage_short'] == "Menstrual")
adata.obs.loc[mask, 'Menstrual_stage_short'] = "Menstrual_fluid"

In [ ]:
# ── Core metadata ─────────────────────────────────────────────────────────────
_meta_cols = [DATASET_COL, ORGAN_COL] + ['Tissue_ROI', 'Organ_part', 'Developmental_stage']
_meta_cols = [c for c in _meta_cols if c in adata.obs.columns]
sc.pl.umap(
    adata,
    color           = _meta_cols,
    ncols           = 1,
    legend_loc      = "right margin",
    legend_fontsize = 7,
    save            = "_01_metadata_acrosslifespan_noHormones_immune_monomacscDC2.png",
)

In [ ]:
plot_obs_facets(adata, "Organ", n_cols=6)

In [ ]:
plot_obs_facets(adata, "Developmental_stage", n_cols=6)

In [ ]:
sc.pl.umap(
    adata,
    color           = "Menstrual_stage_short",
    title           = "UMAP – menstrual stage",
    legend_loc      = "right margin",
    legend_fontsize = 8,
    save            = "_01_menstrual_stage_acrosslifespan_noHormones_immune_monomacscDC2.png",
)

In [ ]:
sc.pl.umap(
    adata,
    color           = "Developmental_stage",
    title           = "UMAP – developmental stage",
    legend_loc      = "right margin",
    legend_fontsize = 8,
    save            = "_01_developmental_stage_acrosslifespan_noHormones_immune_monomacscDC2.png",
)

In [ ]:
# ── Cell cycle phase and doublet score ────────────────────────────────────────
_qc_umap = [c for c in ["phase", "doublet_scores"] if c in adata.obs.columns]
if _qc_umap:
    sc.pl.umap(
        adata,
        color      = _qc_umap,
        ncols      = 2,
        save       = "_02_cellcycle_doublet_acrosslifespan_noHormones_immune_monomacscDC2.png",
    )
else:
    print("No cell-cycle / doublet columns found in obs.")

In [ ]:
sc.pl.umap(
    adata,
    color           = LEIDEN_COL,
    legend_loc      = "on data",
    legend_fontsize = 8,
    title           = "Leiden",
    save            = "_03_leiden_acrosslifespan_noHormones_immune_monomacscDC2.png",
)


# 3. Plot previous annotations

In [ ]:
# ── Prior annotation (optional) ───────────────────────────────────────────────
# Maps a column from a prior annotation CSV into adata.obs (no subsetting).
# The CSV must have a 'barcode' index column. Set to None to skip.
PRIOR_ANN_CSV = os.path.join("/nfs/team292/projects/PanTissue/data/freeze/annotations_author-original/HECA_immune.csv")   # lineage from fetal_02_annotation_fetal
PRIOR_ANN_COL = "immune_subcluster_labels"
PRIOR_BAR_COL = "barcode"


# ── Map prior annotation to obs ───────────────────────────────────────────────
# Reads PRIOR_ANN_CSV and maps PRIOR_ANN_COL into adata.obs by barcode.
# Cells not present in the CSV receive NaN. No subsetting is performed.

if PRIOR_ANN_CSV and os.path.exists(PRIOR_ANN_CSV):
    prior = pd.read_csv(PRIOR_ANN_CSV, index_col=PRIOR_BAR_COL)
    prior.index = prior.index.str.split('-').str[0] + "-1"
    if PRIOR_ANN_COL in prior.columns:
        adata.obs[PRIOR_ANN_COL] = adata.obs_names.map(prior[PRIOR_ANN_COL])
        n_mapped = adata.obs[PRIOR_ANN_COL].notna().sum()
        print(f"Mapped '{PRIOR_ANN_COL}' from {PRIOR_ANN_CSV}")
        print(f"  {n_mapped:,} / {adata.n_obs:,} cells annotated")
        print(adata.obs[PRIOR_ANN_COL].value_counts(dropna=False))
    else:
        print(f"WARNING: column '{PRIOR_ANN_COL}' not found in {PRIOR_ANN_CSV}")
        print(f"  Available columns: {prior.columns.tolist()}")
elif PRIOR_ANN_CSV:
    print(f"WARNING: prior annotation CSV not found:\n  {PRIOR_ANN_CSV}")
else:
    print("No prior annotation CSV specified (PRIOR_ANN_CSV = None) — skipping.")

# Rename
adata.obs = adata.obs.rename(columns={PRIOR_ANN_COL: 'HECA_celltype'})

In [ ]:
sc.pl.umap(
    adata,
    color           = "HECA_celltype",
    legend_loc      = "on data",
    legend_fontsize = 8,
    title           = "HECA_celltype",
    # save            = "_05_lineage_postnatal.png",
)


In [ ]:
# ── Prior annotation (optional) ───────────────────────────────────────────────
# Maps a column from a prior annotation CSV into adata.obs (no subsetting).
# The CSV must have a 'barcode' index column. Set to None to skip.
PRIOR_ANN_CSV = os.path.join("/nfs/team292/projects/PanTissue/data/freeze/annotations_author-original/fallopiantubes_Sanger2026_immune_20160422.csv")   # lineage from fetal_02_annotation_fetal
PRIOR_ANN_COL = "cell_coarse"
PRIOR_BAR_COL = "barcode"


# ── Map prior annotation to obs ───────────────────────────────────────────────
# Reads PRIOR_ANN_CSV and maps PRIOR_ANN_COL into adata.obs by barcode.
# Cells not present in the CSV receive NaN. No subsetting is performed.

if PRIOR_ANN_CSV and os.path.exists(PRIOR_ANN_CSV):
    prior = pd.read_csv(PRIOR_ANN_CSV, index_col=0)
    prior.index = prior.index.astype(str) #+ "-1"
    if PRIOR_ANN_COL in prior.columns:
        adata.obs[PRIOR_ANN_COL] = adata.obs_names.map(prior[PRIOR_ANN_COL])
        n_mapped = adata.obs[PRIOR_ANN_COL].notna().sum()
        print(f"Mapped '{PRIOR_ANN_COL}' from {PRIOR_ANN_CSV}")
        print(f"  {n_mapped:,} / {adata.n_obs:,} cells annotated")
        print(adata.obs[PRIOR_ANN_COL].value_counts(dropna=False))
    else:
        print(f"WARNING: column '{PRIOR_ANN_COL}' not found in {PRIOR_ANN_CSV}")
        print(f"  Available columns: {prior.columns.tolist()}")
elif PRIOR_ANN_CSV:
    print(f"WARNING: prior annotation CSV not found:\n  {PRIOR_ANN_CSV}")
else:
    print("No prior annotation CSV specified (PRIOR_ANN_CSV = None) — skipping.")

# Rename
adata.obs = adata.obs.rename(columns={PRIOR_ANN_COL: 'ECTOPIC_cell_coarse'})

In [ ]:
# 3. Cast to category for plotting
sc.pl.umap(
    adata,
    color           = "ECTOPIC_cell_coarse",
    legend_loc      = "on data",
    legend_fontsize = 8,
    title           = "ECTOPIC_cell_coarse",
    # save            = "_05_lineage_postnatal.png",
)

In [ ]:
# ── Prior annotation (optional) ───────────────────────────────────────────────
# Maps a column from a prior annotation CSV into adata.obs (no subsetting).
# The CSV must have a 'barcode' index column. Set to None to skip.
PRIOR_ANN_CSV = os.path.join("/nfs/team292/projects/PanTissue/data/freeze/annotations_author-original/fallopiantubes_Sanger2026_immune_20160422.csv")   # lineage from fetal_02_annotation_fetal
PRIOR_ANN_COL = "cell_type"
PRIOR_BAR_COL = "barcode"


# ── Map prior annotation to obs ───────────────────────────────────────────────
# Reads PRIOR_ANN_CSV and maps PRIOR_ANN_COL into adata.obs by barcode.
# Cells not present in the CSV receive NaN. No subsetting is performed.

if PRIOR_ANN_CSV and os.path.exists(PRIOR_ANN_CSV):
    prior = pd.read_csv(PRIOR_ANN_CSV, index_col=0)
    prior.index = prior.index.astype(str) #+ "-1"
    if PRIOR_ANN_COL in prior.columns:
        adata.obs[PRIOR_ANN_COL] = adata.obs_names.map(prior[PRIOR_ANN_COL])
        n_mapped = adata.obs[PRIOR_ANN_COL].notna().sum()
        print(f"Mapped '{PRIOR_ANN_COL}' from {PRIOR_ANN_CSV}")
        print(f"  {n_mapped:,} / {adata.n_obs:,} cells annotated")
        print(adata.obs[PRIOR_ANN_COL].value_counts(dropna=False))
    else:
        print(f"WARNING: column '{PRIOR_ANN_COL}' not found in {PRIOR_ANN_CSV}")
        print(f"  Available columns: {prior.columns.tolist()}")
elif PRIOR_ANN_CSV:
    print(f"WARNING: prior annotation CSV not found:\n  {PRIOR_ANN_CSV}")
else:
    print("No prior annotation CSV specified (PRIOR_ANN_CSV = None) — skipping.")

# Rename
adata.obs = adata.obs.rename(columns={PRIOR_ANN_COL: 'ECTOPIC_cell_type'})

In [ ]:
# 3. Cast to category for plotting
sc.pl.umap(
    adata,
    color           = "ECTOPIC_cell_type",
    legend_loc      = "on data",
    legend_fontsize = 8,
    title           = "ECTOPIC_cell_type",
    # save            = "_05_lineage_postnatal.png",
)

In [ ]:
# ── Prior annotation (optional) ───────────────────────────────────────────────
# Maps a column from a prior annotation CSV into adata.obs (no subsetting).
# The CSV must have a 'barcode' index column. Set to None to skip.
PRIOR_ANN_CSV = os.path.join("/nfs/team292/projects/PanTissue/data/freeze/annotations_author-original/fallopiantubes_Sanger2026_immune_20160422.csv")   # lineage from fetal_02_annotation_fetal
PRIOR_ANN_COL = "cell_state"
PRIOR_BAR_COL = "barcode"


# ── Map prior annotation to obs ───────────────────────────────────────────────
# Reads PRIOR_ANN_CSV and maps PRIOR_ANN_COL into adata.obs by barcode.
# Cells not present in the CSV receive NaN. No subsetting is performed.

if PRIOR_ANN_CSV and os.path.exists(PRIOR_ANN_CSV):
    prior = pd.read_csv(PRIOR_ANN_CSV, index_col=0)
    prior.index = prior.index.astype(str) #+ "-1"
    if PRIOR_ANN_COL in prior.columns:
        adata.obs[PRIOR_ANN_COL] = adata.obs_names.map(prior[PRIOR_ANN_COL])
        n_mapped = adata.obs[PRIOR_ANN_COL].notna().sum()
        print(f"Mapped '{PRIOR_ANN_COL}' from {PRIOR_ANN_CSV}")
        print(f"  {n_mapped:,} / {adata.n_obs:,} cells annotated")
        print(adata.obs[PRIOR_ANN_COL].value_counts(dropna=False))
    else:
        print(f"WARNING: column '{PRIOR_ANN_COL}' not found in {PRIOR_ANN_CSV}")
        print(f"  Available columns: {prior.columns.tolist()}")
elif PRIOR_ANN_CSV:
    print(f"WARNING: prior annotation CSV not found:\n  {PRIOR_ANN_CSV}")
else:
    print("No prior annotation CSV specified (PRIOR_ANN_CSV = None) — skipping.")

# Rename
adata.obs = adata.obs.rename(columns={PRIOR_ANN_COL: 'ECTOPIC_cell_state'})

In [ ]:
# 3. Cast to category for plotting
sc.pl.umap(
    adata,
    color           = "ECTOPIC_cell_state",
    legend_loc      = "on data",
    legend_fontsize = 8,
    title           = "ECTOPIC_cell_state",
    # save            = "_05_lineage_postnatal.png",
)

In [ ]:
# ── Prior annotation (optional) ───────────────────────────────────────────────
# Maps a column from a prior annotation CSV into adata.obs (no subsetting).
# The CSV must have a 'barcode' index column. Set to None to skip.
PRIOR_ANN_CSV = os.path.join("/nfs/team292/ja35/To_Christina/immune_annotation_05052026.csv")   # lineage from fetal_02_annotation_fetal
PRIOR_ANN_COL = "cell_type"
# PRIOR_BAR_COL = "barcode"

# ── Map prior annotation to obs ───────────────────────────────────────────────
# Reads PRIOR_ANN_CSV and maps PRIOR_ANN_COL into adata.obs by barcode.
# Cells not present in the CSV receive NaN. No subsetting is performed.

if PRIOR_ANN_CSV and os.path.exists(PRIOR_ANN_CSV):
    prior = pd.read_csv(PRIOR_ANN_CSV, index_col=0)
    # prior.rename(index=lambda x: x.replace('-Sanger_pediatric-Sanger_nonFetal', ''), inplace=True)
    # prior.rename(index=lambda x: x.replace('-Sanger_adult-Sanger_nonFetal', ''), inplace=True)
    # prior.rename(index=lambda x: x.replace('-Sanger_Fetal', ''), inplace=True)
    # prior.index = prior.index.astype(str) + "-1"
    prior.index = prior.index.astype(str).str.replace(r'-.*$', '-1', regex=True)
    if PRIOR_ANN_COL in prior.columns:        
        adata.obs[PRIOR_ANN_COL] = adata.obs_names.map(prior[PRIOR_ANN_COL])
        n_mapped = adata.obs[PRIOR_ANN_COL].notna().sum()
        print(f"Mapped '{PRIOR_ANN_COL}' from {PRIOR_ANN_CSV}")
        print(f"  {n_mapped:,} / {adata.n_obs:,} cells annotated")
        print(adata.obs[PRIOR_ANN_COL].value_counts(dropna=False))
    else:
        print(f"WARNING: column '{PRIOR_ANN_COL}' not found in {PRIOR_ANN_CSV}")
        print(f"  Available columns: {prior.columns.tolist()}")
elif PRIOR_ANN_CSV:
    print(f"WARNING: prior annotation CSV not found:\n  {PRIOR_ANN_CSV}")
else:
    print("No prior annotation CSV specified (PRIOR_ANN_CSV = None) — skipping.")

# Rename
adata.obs = adata.obs.rename(columns={PRIOR_ANN_COL: 'ovaries2026_celltype_UPDATED'})

In [ ]:
# 3. Cast to category for plotting
sc.pl.umap(
    adata,
    color           = "ovaries2026_celltype_UPDATED",
    legend_loc      = "on data",
    legend_fontsize = 8,
    title           = "ovaries2026_celltype_UPDATED",
    # save            = "_05_lineage_postnatal.png",
)

In [ ]:
plot_obs_facets(adata, "ovaries2026_celltype_UPDATED", n_cols=5)

In [ ]:
# ── Prior annotation (optional) ───────────────────────────────────────────────
# Maps a column from a prior annotation CSV into adata.obs (no subsetting).
# The CSV must have a 'barcode' index column. Set to None to skip.
PRIOR_ANN_CSV = os.path.join(f"/nfs/team292/projects/PanTissue/results/temp/02_annotation/immune/postnatal/20260507_postnatal_immune_annotations.csv")
PRIOR_ANN_COL = "celltype"
PRIOR_BAR_COL = "barcode"

# ── Map prior annotation to obs ───────────────────────────────────────────────
# Reads PRIOR_ANN_CSV and maps PRIOR_ANN_COL into adata.obs by barcode.
# Cells not present in the CSV receive NaN. No subsetting is performed.

if PRIOR_ANN_CSV and os.path.exists(PRIOR_ANN_CSV):
    prior = pd.read_csv(PRIOR_ANN_CSV, index_col=PRIOR_BAR_COL)
    # prior.index = prior.index.astype(str) + "-1"
    if PRIOR_ANN_COL in prior.columns:
        adata.obs[PRIOR_ANN_COL] = adata.obs_names.map(prior[PRIOR_ANN_COL])
        n_mapped = adata.obs[PRIOR_ANN_COL].notna().sum()
        print(f"Mapped '{PRIOR_ANN_COL}' from {PRIOR_ANN_CSV}")
        print(f"  {n_mapped:,} / {adata.n_obs:,} cells annotated")
        print(adata.obs[PRIOR_ANN_COL].value_counts(dropna=False))
    else:
        print(f"WARNING: column '{PRIOR_ANN_COL}' not found in {PRIOR_ANN_CSV}")
        print(f"  Available columns: {prior.columns.tolist()}")
elif PRIOR_ANN_CSV:
    print(f"WARNING: prior annotation CSV not found:\n  {PRIOR_ANN_CSV}")
else:
    print("No prior annotation CSV specified (PRIOR_ANN_CSV = None) — skipping.")

# Rename
adata.obs = adata.obs.rename(columns={PRIOR_ANN_COL: 'postnatal_immune_celltype'})

In [ ]:
# 3. Cast to category for plotting
sc.pl.umap(
    adata,
    color           = "postnatal_immune_celltype",
    # legend_loc      = "on data",
    legend_fontsize = 8,
    title           = "postnatal_immune_celltype",
    save            = "_04_postnatal_immune_celltype_mapped_on_acrosslifespan_noHormones_immune_monomacscDC2.png",
)

In [ ]:
# ── Prior annotation (optional) ───────────────────────────────────────────────
# Maps a column from a prior annotation CSV into adata.obs (no subsetting).
# The CSV must have a 'barcode' index column. Set to None to skip.
PRIOR_ANN_CSV = os.path.join(f"/nfs/team292/projects/PanTissue/results/temp/02_annotation/immune/fetal/20260506_fetal_immune_annotations.csv")
PRIOR_ANN_COL = "celltype"
PRIOR_BAR_COL = "barcode"

# ── Map prior annotation to obs ───────────────────────────────────────────────
# Reads PRIOR_ANN_CSV and maps PRIOR_ANN_COL into adata.obs by barcode.
# Cells not present in the CSV receive NaN. No subsetting is performed.

if PRIOR_ANN_CSV and os.path.exists(PRIOR_ANN_CSV):
    prior = pd.read_csv(PRIOR_ANN_CSV, index_col=PRIOR_BAR_COL)
    # prior.index = prior.index.astype(str) + "-1"
    if PRIOR_ANN_COL in prior.columns:
        adata.obs[PRIOR_ANN_COL] = adata.obs_names.map(prior[PRIOR_ANN_COL])
        n_mapped = adata.obs[PRIOR_ANN_COL].notna().sum()
        print(f"Mapped '{PRIOR_ANN_COL}' from {PRIOR_ANN_CSV}")
        print(f"  {n_mapped:,} / {adata.n_obs:,} cells annotated")
        print(adata.obs[PRIOR_ANN_COL].value_counts(dropna=False))
    else:
        print(f"WARNING: column '{PRIOR_ANN_COL}' not found in {PRIOR_ANN_CSV}")
        print(f"  Available columns: {prior.columns.tolist()}")
elif PRIOR_ANN_CSV:
    print(f"WARNING: prior annotation CSV not found:\n  {PRIOR_ANN_CSV}")
else:
    print("No prior annotation CSV specified (PRIOR_ANN_CSV = None) — skipping.")

# Rename
adata.obs = adata.obs.rename(columns={PRIOR_ANN_COL: 'fetal_immune_celltype'})

In [ ]:
# 3. Cast to category for plotting
sc.pl.umap(
    adata,
    color           = "fetal_immune_celltype",
    # legend_loc      = "on data",
    legend_fontsize = 8,
    title           = "fetal_immune_celltype",
    save            = "_04_fetal_immune_celltype_mapped_on_acrosslifespan_noHormones_immune_monomacscDC2.png",
)

In [ ]:
# ── Prior annotation (optional) ───────────────────────────────────────────────
# Maps a column from a prior annotation CSV into adata.obs (no subsetting).
# The CSV must have a 'barcode' index column. Set to None to skip.
PRIOR_ANN_CSV = os.path.join(f"/nfs/team292/projects/PanTissue/results/temp/02_annotation/immune/acrosslifespan/Hormones/20260511_acrosslifespan_immune_annotations.csv")
PRIOR_ANN_COL = "celltype"
PRIOR_BAR_COL = "barcode"

# ── Map prior annotation to obs ───────────────────────────────────────────────
# Reads PRIOR_ANN_CSV and maps PRIOR_ANN_COL into adata.obs by barcode.
# Cells not present in the CSV receive NaN. No subsetting is performed.

if PRIOR_ANN_CSV and os.path.exists(PRIOR_ANN_CSV):
    prior = pd.read_csv(PRIOR_ANN_CSV, index_col=PRIOR_BAR_COL)
    # prior.index = prior.index.astype(str) + "-1"
    if PRIOR_ANN_COL in prior.columns:
        adata.obs[PRIOR_ANN_COL] = adata.obs_names.map(prior[PRIOR_ANN_COL])
        n_mapped = adata.obs[PRIOR_ANN_COL].notna().sum()
        print(f"Mapped '{PRIOR_ANN_COL}' from {PRIOR_ANN_CSV}")
        print(f"  {n_mapped:,} / {adata.n_obs:,} cells annotated")
        print(adata.obs[PRIOR_ANN_COL].value_counts(dropna=False))
    else:
        print(f"WARNING: column '{PRIOR_ANN_COL}' not found in {PRIOR_ANN_CSV}")
        print(f"  Available columns: {prior.columns.tolist()}")
elif PRIOR_ANN_CSV:
    print(f"WARNING: prior annotation CSV not found:\n  {PRIOR_ANN_CSV}")
else:
    print("No prior annotation CSV specified (PRIOR_ANN_CSV = None) — skipping.")

# Rename
adata.obs = adata.obs.rename(columns={PRIOR_ANN_COL: 'acrosslifespan_Hormones_immune_celltype'})

In [ ]:
# 3. Cast to category for plotting
sc.pl.umap(
    adata,
    color           = "acrosslifespan_Hormones_immune_celltype",
    # legend_loc      = "on data",
    legend_fontsize = 8,
    title           = "acrosslifespan_Hormones_immune_celltype",
    save            = "_04_acrosslifespan_Hormones_immune_celltype_mapped_on_acrosslifespan_noHormones_immune_monomacscDC2.png",
)

In [ ]:
adata.obs.head(1)

# 4. Optional: re-cluster specific Leiden clusters

Duplicate the cell below for each cluster that needs splitting.

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['3']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['4']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['2']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['14']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['8']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['1']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['26']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['33']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['37']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['31']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['25']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['0']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['0']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['34']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['32']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['10']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['62']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['27']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
# ── Template — duplicate this cell for each cluster to re-split ───────────────

sc.tl.leiden(adata, resolution=0.3, restrict_to=('leiden', ['68']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden_R'] = adata.obs['leiden_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs['leiden_R'].cat.categories))]
)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', legend_fontsize=9)

adata.obs['leiden'] = adata.obs['leiden_R']

In [ ]:
adata.obs['leiden'].unique()

In [ ]:
plot_obs_facets(adata, "leiden", n_cols=6)

In [ ]:
sc.pl.umap(
    adata,
    color           = LEIDEN_COL,
    legend_loc      = "on data",
    legend_fontsize = 4,
    title           = "Leiden",
    palette         = palette,
    save            = "_03_leiden_acrosslifespan_noHormones_immune_monomacscDC2.png",
)

# 5. TF-IDF: quick marker identification

In [ ]:
ANNOT_COL = "celltype"

In [ ]:
mask = ~adata.var_names.str.startswith("DEPRECATED_")
bdata = adata[:, mask].copy()

print(f"Num genes removed: {(adata.n_vars)-(bdata.n_vars)}")

In [ ]:
from scipy.sparse import csr_matrix, issparse
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

def quick_markers(
    adata,
    cluster_key,
    cell_groups=None,
    layer=None,
    n_markers=10,
    fdr=0.01,
    express_cut=0.9,
    r_output=False
):
    """
    Identifies top N markers for each cluster in an AnnData object
    using a TF-IDF-based strategy similar to SoupX.

    Parameters
    ----------
    adata : AnnData
        Annotated data matrix.

    cluster_key : str
        Column in adata.obs containing cluster labels.

    cell_groups : list, optional
        Subset of clusters to analyze.

    layer : str, optional
        Layer to use instead of adata.X.

    n_markers : int
        Number of markers per cluster.

    fdr : float
        FDR threshold.

    express_cut : float
        Expression threshold for binarization.

    r_output : bool
        Return SoupX-style column names.

    Returns
    -------
    pandas.DataFrame
    """

    # ------------------------------------------------------------------
    # Subset cells if requested
    # ------------------------------------------------------------------

    if cell_groups is not None:
        adata_ = adata[adata.obs[cluster_key].isin(cell_groups)].copy()
    else:
        adata_ = adata.copy()

    # ------------------------------------------------------------------
    # Ensure categorical consistency
    # ------------------------------------------------------------------

    adata_.obs[cluster_key] = (
        adata_.obs[cluster_key]
        .astype("category")
        .cat.remove_unused_categories()
    )

    cluster_cat = adata_.obs[cluster_key].cat
    cluster_names = cluster_cat.categories

    clusters = cluster_cat.codes.to_numpy()

    unique_clusters = np.unique(clusters)

    # ------------------------------------------------------------------
    # Expression matrix
    # ------------------------------------------------------------------

    if layer is not None:
        toc = (
            csr_matrix(adata_.layers[layer])
            if not issparse(adata_.layers[layer])
            else adata_.layers[layer]
        )
    else:
        toc = (
            csr_matrix(adata_.X)
            if not issparse(adata_.X)
            else adata_.X
        )

    # Binarize
    toc_bin = (toc > express_cut).astype(int)

    # ------------------------------------------------------------------
    # Cluster counts
    # ------------------------------------------------------------------

    cl_counts = np.asarray([
        np.sum(clusters == cl)
        for cl in unique_clusters
    ]).reshape(-1, 1)

    # ------------------------------------------------------------------
    # Observed frequencies
    # ------------------------------------------------------------------

    n_obs = np.asarray([
        np.asarray(
            toc_bin[clusters == cl, :].sum(axis=0)
        ).flatten()
        for cl in unique_clusters
    ])

    n_tot = n_obs.sum(axis=0)

    # Avoid divide-by-zero
    n_tot_safe = np.where(n_tot == 0, 1, n_tot)

    # ------------------------------------------------------------------
    # TF-IDF
    # ------------------------------------------------------------------

    tf = n_obs / cl_counts

    idf = np.log(len(clusters) / n_tot_safe)

    tf_idf = tf * idf

    # ------------------------------------------------------------------
    # Additional metrics
    # ------------------------------------------------------------------

    gene_freq_outside_cluster = (
        (n_tot - n_obs)
        / np.maximum(len(clusters) - cl_counts, 1)
    )

    gene_freq_global = n_tot / len(clusters)

    # ------------------------------------------------------------------
    # Second-best TF cluster
    # ------------------------------------------------------------------

    second_best_tf = np.zeros_like(tf)

    second_best_cluster_idx = np.zeros(
        tf.shape[1],
        dtype=int
    )

    for gene_idx in range(tf.shape[1]):

        tf_scores = tf[:, gene_idx]

        if len(tf_scores) > 1:
            second_best_idx = np.argsort(tf_scores)[-2]
        else:
            second_best_idx = 0

        second_best_tf[:, gene_idx] = tf_scores[second_best_idx]

        second_best_cluster_idx[gene_idx] = second_best_idx

    # ------------------------------------------------------------------
    # Hypergeometric p-values
    # ------------------------------------------------------------------

    p_values = np.array([
        hypergeom.sf(
            n_obs[i] - 1,
            len(clusters),
            n_tot,
            cl_counts[i]
        )
        for i in range(len(unique_clusters))
    ])

    # ------------------------------------------------------------------
    # FDR correction
    # ------------------------------------------------------------------

    p_flat = p_values.flatten()

    reject, q_flat, _, _ = multipletests(
        p_flat,
        alpha=fdr,
        method='fdr_bh'
    )

    q_values = q_flat.reshape(p_values.shape)

    # ------------------------------------------------------------------
    # Select top markers
    # ------------------------------------------------------------------

    top_markers = {
        cl: []
        for cl in unique_clusters
    }

    for gene_idx in range(tf_idf.shape[1]):

        for cl in unique_clusters:

            q_val = q_values[cl, gene_idx]

            if not np.isnan(q_val) and q_val < fdr:

                top_markers[cl].append(
                    (gene_idx, tf_idf[cl, gene_idx])
                )

    # Sort and keep top N
    for cl in top_markers:

        top_markers[cl].sort(
            key=lambda x: x[1],
            reverse=True
        )

        top_markers[cl] = [
            gene_idx
            for gene_idx, _
            in top_markers[cl][:n_markers]
        ]

    # ------------------------------------------------------------------
    # Build output table
    # ------------------------------------------------------------------

    marker_data = []

    for cl, markers in top_markers.items():

        for gene_idx in markers:

            second_best_cl = second_best_cluster_idx[gene_idx]

            marker_data.append({

                'gene':
                    adata_.var_names[gene_idx],

                'cluster':
                    cluster_names[cl],

                'tf':
                    tf[cl, gene_idx],

                'idf':
                    idf[gene_idx],

                'tf_idf':
                    tf_idf[cl, gene_idx],

                'gene_frequency_outside_cluster':
                    gene_freq_outside_cluster[cl, gene_idx],

                'gene_frequency_global':
                    gene_freq_global[gene_idx],

                'second_best_tf':
                    second_best_tf[cl, gene_idx],

                'second_best_cluster':
                    cluster_names[second_best_cl],

                'pval':
                    p_values[cl, gene_idx],

                'qval':
                    q_values[cl, gene_idx]
            })

    markers = pd.DataFrame(marker_data)

    # ------------------------------------------------------------------
    # Empty result handling
    # ------------------------------------------------------------------

    if markers.shape == (0, 0):

        markers = pd.DataFrame(columns=[
            'gene',
            'cluster',
            'tf',
            'idf',
            'tf_idf',
            'gene_frequency_outside_cluster',
            'gene_frequency_global',
            'second_best_tf',
            'second_best_cluster',
            'pval',
            'qval'
        ])

    # ------------------------------------------------------------------
    # SoupX-compatible output
    # ------------------------------------------------------------------

    if r_output:

        cols = [
            'gene',
            'cluster',
            'tf',
            'gene_frequency_outside_cluster',
            'second_best_tf',
            'gene_frequency_global',
            'second_best_cluster',
            'tf_idf',
            'idf',
            'qval'
        ]

        markers = markers[cols]

        markers.columns = [
            'gene',
            'cluster',
            'geneFrequency',
            'geneFrequencyOutsideCluster',
            'geneFrequencySecondBest',
            'geneFrequencyGlobal',
            'secondBestClusterName',
            'tfidf',
            'idf',
            'qval'
        ]

    return markers

In [ ]:
# Run TF-IDF-based marker identification on clustered data
markers = quick_markers(
    bdata,
    cluster_key=ANNOT_COL,
    cell_groups=[
        "Immune_Mac_LYVE1",
        "Immune_Mac_EREG",
        "Immune_Mac_TREM2hi",
        "Immune_Mac_TREM2hi_FCGBPhi",
        "Immune_Mac_TREM2hi_DHRS9"
    ],
    layer=None,
    n_markers=10,
    r_output=True
)

# markers.to_csv(f"{OUTPUT_DIR}tf-idf_markers_{ANNOT_COL}_acrosslifespan_immune.csv")
markers.head()

In [ ]:
def plot_tfidf(adata,
               markers: pd.DataFrame,
               top_n: int=10,
               group: str='leiden',
               plot_unique: bool=False,
               outdir: str=None,
               filename: str=None,
               save: bool=False):
    
    """
    Visualises top TF-IDF-based markers per cluster in a dotplot.
    
    Parameters
    ----------
    adata : AnnData
        The annotated data matrix of shape `n_obs` × `n_vars`. Rows correspond to cells and columns to genes
    markers : pd.DataFrame
        A :class:`pd.DataFrame` that is output from tfidf_markers() (see function for structure)
    top_n : int, default=10
        How many top TF-IDF markers to plot per cluster. Default is 10
    group : str, default='leiden'
        Column name in `adata.obs` to group top markers by. Defaults to Leiden (unannotated clusters generated
        by Leiden algorithm)
    plot_unique : bool, default=False
        Whether to plot an unordered set of unique top_n markers over all clusters. Default is `False`, outputting
        a dotplot grouped by cluster
    outdir, save : optional
        Figure saving parameters
    """
    
    # Establish unique set of markers and dictionary containing cluster-labeled top_n markers for plotting
    unique_markers = set()
    all_markers = dict()
    for cluster, df in markers.groupby('cluster'): # Iterating through a GroupBy object allows you to extract group key (cluster number) and subset of the DataFrame that corresponds to that group
        top_markers = [g for g in df['gene']][:top_n] # Extract gene names of top_n markers for given cluster
        all_markers[cluster] = top_markers # Append top_n markers to corresponding cluster in markers dictionary
        tmp_genes = set(top_markers) # Get unique list of genes over all top markers for all clusters
        unique_markers = unique_markers.union(tmp_genes)
    unique_markers = sorted(unique_markers)
    
    # Plot dotplot
    with plt.rc_context({"font.size": 11, "figure.dpi": 80, "figure.figsize": (6, 2)}):
        if plot_unique == True:
            dp = sc.pl.dotplot(adata, unique_markers, return_fig=True, standard_scale='var')
        else:
            dp = sc.pl.dotplot(adata, all_markers, groupby=group, return_fig=True, standard_scale='var')

        dp.style(cmap='OrRd', dot_edge_color='black', dot_edge_lw=0.8)
        if save is True and outdir and filename:
            dp.savefig(f"{outdir}tf-idf_{filename}.pdf")
    dp.show() if save==False else None  

In [ ]:
# Plot the markers dotplot and, if desired, save as PDF
plot_tfidf(
    bdata,
    markers,
    top_n=10,
    group=ANNOT_COL,
    outdir=os.path.join(OUTPUT_DIR, "figures/"),
    filename=f'{ANNOT_COL}_Macs_acrosslifespan_noHormones_immune_monomacscDC2',
    save=True
)

# 6. Plot known markers

## Normalise for visualisation (raw counts frozen)

In [ ]:
adata.raw = adata   # freeze raw counts
sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
sc.pp.log1p(adata)
print("Raw counts frozen in adata.raw. adata.X now holds log-normalised expression.")

## Load marker genes

In [ ]:
spec = importlib.util.spec_from_file_location("markers", MARKERS_PY)
markers_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(markers_mod)

In [ ]:
# ── 1. Broad lineage markers ──────────────────────────────────────────────────
markers_lineage = getattr(markers_mod, "markers_lineage", {})
markers_lineage_avail = get_available_markers(markers_lineage, "Lineage markers", adata)

In [ ]:
# ── 4. Macrophage postnatal markers, all tissues ──────────────────────────────
if hasattr(markers_mod, "markers_immune_macros"):
    markers_immMac = markers_mod.markers_immune_macros
    markers_immMac_avail = get_available_markers(markers_immMac, "Postnatal macrophage markers", adata)
else:
    print("WARNING: 'markers_immune_macros' not found.")
    markers_immMac_avail = {}

In [ ]:
# ── 11. Manual postnatal markers, after annotation ─────────────────────────────
markers_general = {
    'Circulating': ['SELL'], 
    'Myeloid': ['CD14', 'CD68'],
    'Lymphoid': ['CD3D', 'CD3G', 'NKG7', 'CD79A']
}

markers_prog = {
    'HPC': ['CD34', 'SPINK2', 'PROM1', 'HOPX', 'FLT3'],
    'MEMP': ['GATA1', 'GATA2', 'FCER1A', 'CNRIP1', 'HBD'],
    'NMP/Neutrophils': ['MPO', 'PRTN3', 'AZU1', 'ELANE', 'CTSG']
}

markers_myeloid = {
    'Early/MidErythroid': ['GYPA', 'KLF1', 'ALAS2', 'TFRC'], # Cycling, TRFChi - erythroblasts
    'LateErythroid': ['HBA1', 'BPGM'],
    'Mast': ['CPA3', 'KIT'],
    'Megakaryocytes': ['PF4', 'ITGA2B'],
    'MO': ['S100A12', 'FCN1', 'LILRA5', 'S100A9'],
    # 'MO_Cycling': [],
    'Mac_LYVE1+': ['LYVE1', 'FOLR2', 'SELENOP', 'LILRB5', 'SPP1', 'PID1'],
    'Mac_TREM2hi': ['TREM2', 'CD9', 'LGALS3', 'APOE', 'DHRS9', 'CX3CR1', 'OLFML3'], # 18 is CX3CR1/OLFML3hi subtype
    'Mac_NR1H3hi': ['NR1H3', 'CD5L']
    # 'Mac_Cycling': [],
}

markers_DC = {
    'cDC1': ['CLEC9A', 'XCR1', 'IDO1'],
    'cDC2': ['CLEC10A', 'CD1C'],
    'pDC': ['LILRA4', 'SCT', 'CLEC4C']
}

markers_lymphoid = {
    'Common_B': ['CD79A', 'PAX5', 'CD19'],
    'Early_B_Dev': ['IL7R', 'RAG1', 'RAG2', 'VPREB1', 'IGLL1', 'CD24', 'MME', 'EBF1', 'TCF4'],
    'Mid_B_Dev': ['CCDC191', 'LEF1', 'IKZF2', 'ZEB2'],
    'Late_B_Dev': ['MS4A1', 'BANK1', 'BLK', 'IGHM', 'IGHD', 'IGLC1', 'IRF8'],
    'ILC3': ['RORC', 'TOX2', 'IL1R1'],
    'General_T': ['CD8A', 'CD8B', 'CD4', 'TRAC'],
    'NaiveT': ['TCF7', 'MAL', 'THEMIS'],
    'EarlyT': ['GATA3', 'LTB', 'IL32' ,'CD44', 'TRBC1'],
    'Immune_Treg': ['FOXP3', 'CTLA4', 'IL2RA'],
    'gdT': ['TRDV2', 'TRGV9', 'IL23R'],
    'Immune_T/NK': ['NCAM1', 'FCGR3A', 'KLRC1', 'KLRD1']
}

In [ ]:
markers = {
    'Neutrophils': ['ALPL', 'CXCR2', 'FCGR3B'],
    'cDC2': ['CD1C', 'CD1E', 'FCER1A', 'CD207'],
    
    'MO_CD14': ['S100A12', 'PLCB1'],
    'MO_CD16': ['FCGR3A', 'PTP4A3', 'CDKN1C'],
    
    'Mac_LYVE1': ['LYVE1', 'FOLR2', 'SELENOP'],
    'Mac_TREM2': ['TREM2']
    'Mac_EREG': [],
    
    'Cycling': ['MKI67', 'PCLAF']
}

## Dotplots — markers × clusters

In [ ]:
# ── Broad lineage markers (doublet / contamination check) ─────────────────────
sc.pl.dotplot(
    bdata,
    var_names      = ['TREM2', 'APOE', 'CD9', 'SPP1', 'OLR1', 'CH25H', 'SORL1', 'APOC2', 'ALDH2'],
    groupby        = 'celltype',
    # use_raw        = True,
    standard_scale = "var",
    colorbar_title = "Mean expr\n(scaled)",
    # save           = "04a_leiden_linmarkers_acrosslifespan_immune_monomacscDC2.png",
)

In [ ]:
# ── Broad lineage markers (doublet / contamination check) ─────────────────────
sc.pl.dotplot(
    adata,
    var_names      = markers_lineage_avail,
    groupby        = LEIDEN_COL,
    # use_raw        = True,
    standard_scale = "var",
    colorbar_title = "Mean expr\n(scaled)",
    save           = "04a_leiden_linmarkers_acrosslifespan_immune_monomacscDC2.png",
)

In [ ]:
# ── postnatal immune cell type markers ────────────────────────────────────────
if markers_immMac_avail:
    sc.pl.dotplot(
        adata,
        var_names      = markers_immMac_avail,
        groupby        = LEIDEN_COL,
        # use_raw        = True,
        standard_scale = "var",
        colorbar_title = "Mean expr\n(scaled)",
        save           = "04b_leiden_immMacmarkers_acrosslifespan_immune_monomacscDC2.png",
    )
else:
    print("Skipping dotplot — markers not loaded.")

# 7. Manual cell type annotation

Fill in Leiden cluster IDs (integers) for each postnatal immune cell type.  
Leave empty lists `[]` for cell types not present or not yet identified.

In [ ]:
# ── Fill in Leiden IDs as integers ────────────────────────────────────────────
# Example:  'Epi_FT_Cil': [3, 7]
celltype_to_leiden = {
    'Immune_Neutrophils':[26],
    'Immune_cDC2':[40,41,43,44,46,47],
    'Immune_cDC2_CD207hi':[37,38,39,45],                # Separated based on CD207 expression (violin plot Loupe)
    'Immune_tolDC':[42],                                # PRDM16, RORC+ DC - Fu et al., 2025 Nature

    'Immune_MO_Classical':[
        25,27,28,29,31,33,35,52,58,59,60,62,63
    ],
    'Immune_MO_NonClassical':[72],
    'Immune_Mac_LYVE1hi':[
        7,8,9,10,11,13,14,15,24,32,34,53,54,55
    ],
    'Immune_Mac_Transitional':[
        4,6,16,18,20,21,56
    ],
    'Immune_uftLAM':[48,49,51],
    'Immune_oLAM':[17],
    'Immune_uMac_Inf':[
        23,36,57,65,66,67,68,69,70
    ],
        
    'Immune_Mac_Cyc':[0,2,3,5],

        'doublet': [1,64,71],
        # 'soup': [],
        'lowQC': [12,50],
        'donor_specific': [19,22,30,61],
        # 'unknown': [],
}


# Convert integers to strings
celltype_to_leiden = {k: [str(v) for v in vals] for k, vals in celltype_to_leiden.items()}

# ── Validate Leiden IDs ───────────────────────────────────────────────────────
_all_assigned = [lid for lids in celltype_to_leiden.values() for lid in lids]
_all_leiden   = sorted(adata.obs[LEIDEN_COL].unique().tolist())
_dups         = [x for x in _all_assigned if _all_assigned.count(x) > 1]
_unassigned   = [x for x in _all_leiden if x not in _all_assigned]

if _dups:
    print(f"WARNING — clusters assigned to multiple cell types: {sorted(set(_dups))}")
if _unassigned:
    print(f"INFO    — clusters not yet assigned              : {_unassigned}")
if not _dups and not _unassigned:
    print("All clusters assigned, no duplicates.")

In [ ]:
# Invert: leiden_id → celltype
leiden_to_celltype = {
    lid: celltype
    for celltype, lids in celltype_to_leiden.items()
    for lid in lids
}

adata.obs["celltype"] = (
    adata.obs[LEIDEN_COL]
    .map(leiden_to_celltype)
    .astype("category")
)

print("Celltype distribution:")
print(adata.obs["celltype"].value_counts())

In [ ]:
# Order categories
celltype_order = list(celltype_to_leiden.keys())

adata.obs["celltype"] = adata.obs["celltype"].cat.reorder_categories(
    celltype_order,
    ordered=True
)

print("Celltype distribution:")
print(adata.obs["celltype"].value_counts())

In [ ]:
# celltype_leiden = "<celltype>_<leiden>"
adata.obs["celltype_leiden"] = (
    adata.obs["celltype"].astype(str)
    + "_"
    + adata.obs[LEIDEN_COL].astype(str)
).astype("category")

print("Celltype_leiden categories:", sorted(adata.obs["celltype_leiden"].cat.categories.tolist()))

In [ ]:
adata.obs["celltype_leiden"].isna().sum()

## Validation UMAPs

In [ ]:
ANNOT_COL = "celltype"

sc.pl.umap(
    adata,
    color           = ANNOT_COL,
    legend_loc      = "on data",
    legend_fontsize = 2,
    title           = ANNOT_COL,
    save            = f"_05_{ANNOT_COL}_acrosslifespan_noHormones_immune_monomacscDC2.pdf",
)


# 8. Save

In [ ]:
adata.obs.head(1)

In [ ]:
# adata.obs_names = adata.obs["Loupe_barcodes"]
adata.write_h5ad(FORANNOTION_H5AD)
print(f"Annotated object saved → {FORANNOTION_H5AD}")

In [ ]:
adata.obs_names = adata.obs["sample_id_barcode_copy"]
adata.obs.index.name = "barcode"
adata.write_h5ad(ANNOTATED_H5AD)
print(f"Annotated object saved → {ANNOTATED_H5AD}")

In [ ]:
print("Organ distribution across all cells:")
print(adata.obs["Organ"].value_counts())

## Save obs for annotations

In [ ]:
# ── Save annotations ────────────────────────────────—————————————
adata.obs.to_csv(
    f'{OUTPUT_DIR}20260512_acrosslifespan_noHormones_immune_monomacscDC2_annotations.csv',
    index=True,
    index_label='barcode'
)

# 9. Barplots

In [ ]:
BARPLOT_COLS = [
    "celltype",
    # LEIDEN_COL
]

PROP_COL = "Developmental_stage"

_n_groups = adata.obs[PROP_COL].nunique()
_height   = max(0.25, _n_groups * 0.25)

for col in BARPLOT_COLS:
    if col not in adata.obs.columns:
        print(f"Skipping '{col}' — not found in obs.")
        continue
    Barplot(PROP_COL, adata, var=col, height=8)
    Barplot(col, adata, var=PROP_COL, height=8)
    
    # plt.savefig(f"{OUTPUT_DIR}figures/barplot_{col}_{PROP_COL}_acrosslifespan_noHormones_immune_monomacscDC2.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
BARPLOT_COLS = [
    "celltype",
    # LEIDEN_COL
]

PROP_COL = "Organ"

_n_groups = adata.obs[PROP_COL].nunique()
_height   = max(0.25, _n_groups * 0.25)

for col in BARPLOT_COLS:
    if col not in adata.obs.columns:
        print(f"Skipping '{col}' — not found in obs.")
        continue
    Barplot(PROP_COL, adata, var=col, height=8)
    
    plt.savefig(f"{OUTPUT_DIR}figures/barplot_{col}_{PROP_COL}_acrosslifespan_noHormones_immune_monomacscDC2.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
BARPLOT_COLS = [
    "celltype",
    # LEIDEN_COL
]

PROP_COL = "Organ_part"

_n_groups = adata.obs[PROP_COL].nunique()
_height   = max(0.25, _n_groups * 0.25)

for col in BARPLOT_COLS:
    if col not in adata.obs.columns:
        print(f"Skipping '{col}' — not found in obs.")
        continue
    Barplot(PROP_COL, adata, var=col, height=_height)
    
    plt.savefig(f"{OUTPUT_DIR}figures/barplot_{col}_{PROP_COL}_acrosslifespan_noHormones_immune_monomacscDC2.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
BARPLOT_COLS = [
    # "celltype",
    LEIDEN_COL
]

PROP_COL = "Donor_id"

_n_groups = adata.obs[PROP_COL].nunique()
_height   = max(0.25, _n_groups * 0.25)

for col in BARPLOT_COLS:
    if col not in adata.obs.columns:
        print(f"Skipping '{col}' — not found in obs.")
        continue
    Barplot(PROP_COL, adata, var=col, height=_height)
    
    # plt.savefig(f"{OUTPUT_DIR}figures/barplot_{col}_{PROP_COL}_acrosslifespan_noHormones_immune_monomacscDC2.png", dpi=300, bbox_inches="tight")
    plt.show()